# Compress any HuggingFace model to a standalone GGUF Q4_K_M file

This notebook takes any instruct model from HuggingFace and produces a **standalone compressed GGUF file** that:

- Is roughly **25-40% of the original size**
- Loads **on its own**, with **no base model required** at inference time
- Runs in `llama.cpp`, `llama-cpp-python`, Ollama, LM Studio, kobold.cpp, etc.
- Preserves the original chat template (baked into the GGUF metadata)

### Why this matters

Many "compression" pipelines (LoRA deltas, sparse-diff patches, low-rank adapters) still require you to download the original model at runtime. **GGUF quantization is true standalone compression** — the only file you ship is the compressed one.

### What this notebook does

1. Install `llama.cpp` from source + Python deps
2. Download a HuggingFace model you specify
3. Convert it to GGUF (F16 intermediate)
4. Quantize to **Q4_K_M** (best quality/size balance per llama.cpp benchmarks)
5. Verify the compressed file loads standalone and generates text
6. Show before/after size comparison
7. Provide a download link for the compressed file

### What this notebook does NOT do

- It does **not** keep the original model around at inference time.
- It does **not** require the base model file after quantization.
- It does **not** use any adapter / delta / patch layer.

---

## 1. Configuration

Set the model you want to compress. Defaults to `Qwen/Qwen2.5-0.5B-Instruct` (small + fast + real instruct model, proven to work). Change `MODEL_ID` to any HF instruct model that llama.cpp supports (Llama, Mistral, Qwen, Gemma, Phi, Yi, etc.).

In [ ]:
# @title Configuration { display-mode: "form" }

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # @param {type:"string"}
QUANT_TYPE = "Q4_K_M"  # @param ["Q4_K_M", "Q4_K_S", "Q5_K_M", "Q5_K_S", "Q8_0", "Q3_K_M", "Q2_K"]
UPLOAD_TO_HF = False  # @param {type:"boolean"}
HF_REPO_ID = ""  # @param {type:"string"}

# Computed names
MODEL_SHORT = MODEL_ID.split("/")[-1]
HF_DIR = f"/content/models/{MODEL_SHORT}"
F16_GGUF = f"/content/{MODEL_SHORT}-F16.gguf"
COMPRESSED_GGUF = f"/content/{MODEL_SHORT}-{QUANT_TYPE}.gguf"

print(f"Model:        {MODEL_ID}")
print(f"Quant type:   {QUANT_TYPE}")
print(f"HF dir:       {HF_DIR}")
print(f"Compressed:   {COMPRESSED_GGUF}")

## 2. Install build tools + clone + compile llama.cpp

We build `llama-quantize` from source because the `convert_hf_to_gguf.py` script and the quantizer need to match versions. ~5 minutes on a free Colab CPU instance.

In [ ]:
!apt-get install -y cmake build-essential git wget 2>&1 | tail -3
!pip install -q huggingface_hub transformers accelerate sentencepiece protobuf torch

import os
os.chdir("/content")
if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

os.chdir("/content/llama.cpp")
!pip install -q -r requirements/requirements-convert_hf_to_gguf.txt

# Build only the binaries we need (quantize + cli + bench)
!mkdir -p build && cd build && cmake .. \
    -DGGML_NATIVE=ON \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=OFF \
    -DLLAMA_BUILD_SERVER=OFF 2>&1 | tail -3
!cd build && cmake --build . --target llama-quantize llama-cli llama-bench -j 2>&1 | tail -3

print("\n=== llama.cpp built ===")
!ls -la /content/llama.cpp/build/bin/ | grep -E 'quantize|cli|bench' | head

## 3. Download the original HuggingFace model

This is the **only** step that touches the uncompressed model. After step 4 produces the compressed GGUF, you can delete this directory — the compressed file is fully self-contained.

In [ ]:
from huggingface_hub import snapshot_download
import os

os.makedirs("/content/models", exist_ok=True)

print(f"Downloading {MODEL_ID} ...")
local_dir = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=HF_DIR,
)
print(f"Saved to: {local_dir}")

# Measure original size
total = 0
for root, _, files in os.walk(HF_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f"Original model size: {total / 1024 / 1024:.1f} MiB ({total / 1024 / 1024 / 1024:.2f} GiB)")

## 4. Convert HF \u2192 GGUF (F16 intermediate)

Lossless conversion. The F16 GGUF is the same weights as the original, just in llama.cpp's container format.

In [ ]:
import os
os.chdir("/content/llama.cpp")

#!python convert_hf_to_gguf.py --help | head -30
!python convert_hf_to_gguf.py "{HF_DIR}" --outfile "{F16_GGUF}" --outtype f16

f16_size = os.path.getsize(F16_GGUF)
print(f"\nF16 GGUF size: {f16_size / 1024 / 1024:.1f} MiB")

## 5. Quantize F16 \u2192 {Q4_K_M} (standalone compressed file)

This is the standalone compressed model. After this step, you can delete both the HF directory and the F16 GGUF — the output of this cell is self-contained.

In [ ]:
import os
os.chdir("/content/llama.cpp")

!./build/bin/llama-quantize "{F16_GGUF}" "{COMPRESSED_GGUF}" {QUANT_TYPE}

comp_size = os.path.getsize(COMPRESSED_GGUF)
print(f"\nCompressed ({QUANT_TYPE}) GGUF size: {comp_size / 1024 / 1024:.1f} MiB")
print(f"Compression ratio: {comp_size / f16_size * 100:.1f}% of F16")

## 6. Free up space: delete the originals

From this point forward, **only the compressed GGUF is needed**. We delete the HF directory and the F16 intermediate to prove it.

In [ ]:
import shutil, os

# Delete the original HF model directory
if os.path.exists(HF_DIR):
    shutil.rmtree(HF_DIR)
    print(f"Deleted: {HF_DIR}")

# Delete the F16 intermediate
if os.path.exists(F16_GGUF):
    os.remove(F16_GGUF)
    print(f"Deleted: {F16_GGUF}")

print(f"\nRemaining model file: {COMPRESSED_GGUF}")
print(f"Size: {os.path.getsize(COMPRESSED_GGUF) / 1024 / 1024:.1f} MiB")

## 7. Smoke test: load ONLY the compressed file and generate text

This proves the compressed file is truly standalone. We use `llama-cpp-python`, which only opens the compressed GGUF — no HF model is loaded.

In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=COMPRESSED_GGUF,
    n_ctx=2048,
    n_threads=2,
    n_gpu_layers=0,  # set to 99 if you have a GPU runtime
    verbose=False,
)
print(f"Load time: {time.time()-t0:.2f}s")

t0 = time.time()
out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "In one sentence, explain why the sky is blue."},
    ],
    max_tokens=64,
    temperature=0.7,
)
print(f"Generate time: {time.time()-t0:.2f}s")
print(f"\nResponse: {out['choices'][0]['message']['content']}")
print(f"Usage: {out.get('usage')}")

## 8. (Optional) Upload compressed file to your own HuggingFace repo

Skip this if you don't want to publish. If you do, set `UPLOAD_TO_HF = True` and `HF_REPO_ID = "your-username/your-repo"` in the configuration cell, then provide your HF token.

In [ ]:
if UPLOAD_TO_HF:
    from huggingface_hub import HfApi, create_repo
    api = HfApi()
    create_repo(HF_REPO_ID, exist_ok=True, repo_type="model")
    api.upload_file(
        path_or_fileobj=COMPRESSED_GGUF,
        path_in_repo=os.path.basename(COMPRESSED_GGUF),
        repo_id=HF_REPO_ID,
    )
    print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Upload disabled. Skipping.")
    print(f"Compressed file is at: {COMPRESSED_GGUF}")

## 9. Download the compressed file to your machine

Colab provides a built-in download link.

In [ ]:
from google.colab import files
files.download(COMPRESSED_GGUF)

## 10. Next steps

Now you have a standalone compressed `.gguf` file. To use it in your own project:

```bash
pip install llama-cpp-python
```

```python
from llama_cpp import Llama

llm = Llama(model_path="path/to/your-model-Q4_K_M.gguf", n_ctx=2048, n_threads=4)
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Hello!"}],
    max_tokens=128,
)
print(out["choices"][0]["message"]["content"])
```

No transformers, no PyTorch, no HuggingFace model directory, no tokenizer files. Just the one `.gguf` file.

---

### Quantization type reference

| Type   | Approx size (vs F16) | Quality   | When to use |
|--------|----------------------|-----------|-------------|
| Q8_0   | ~50%                 | Best      | When you can afford the size |
| Q6_K   | ~45%                 | Excellent | Production, balanced |
| Q5_K_M | ~40%                 | Very good | Recommended sweet spot |
| **Q4_K_M** | **~35%**         | **Good**  | **Recommended default** |
| Q4_K_S | ~33%                 | Good      | Smaller variant of Q4_K_M |
| Q3_K_M | ~28%                 | Acceptable| Tight size budget |
| Q2_K   | ~22%                 | Noticeable loss | Last resort |